In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation"
%pwd

/content/drive/MyDrive/Colab Notebooks/clothes-segmentation


'/content/drive/MyDrive/Colab Notebooks/clothes-segmentation'

In [4]:
import sys
PROJECT_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation"
)
sys.path.append(PROJECT_PATH)

In [6]:
%run "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/scripts/data_download.py"

100%|██████████| 616M/616M [00:18<00:00, 34.2MB/s]

Extracting files...


Images: 1000, Masks: 1000


In [12]:
import importlib
import configs
importlib.reload(configs)

<module 'configs' from '/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/configs.py'>

In [13]:
import configs
print(configs.__file__)          # confirms which file was actually loaded
print(configs.BaseConfig.__dict__.keys())  # confirms what attributes it actually has
print(configs.BaseConfig.__dict__.values())  # confirms what attributes it actually has

/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/configs.py
dict_keys(['__module__', '__firstlineno__', 'DATASET_URL', 'PROJECT_ROOT', 'DATA_ROOT', 'IMG_SIZE', 'VAL_SIZE', 'TEST_SIZE', 'RANDOM_SEED', 'NUM_AUGMENTED_COPIES', 'NUM_CLASSES', 'MODEL_ROOT', 'UNET_FEATURES', 'UNET_PP_FEATURES', 'DICE_WEIGHT', 'DEFAULT_EPOCHS', 'DEFAULT_BATCH_SIZE', 'DEFAULT_LR', '__static_attributes__', '__dict__', '__weakref__', '__doc__'])
dict_values(['configs', 2, 'rajkumarl/people-clothing-segmentation', '/content/drive/MyDrive/Colab Notebooks/clothes-segmentation', '/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data', 256, 0.1, 0.1, 42, 2, 9, '/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models', [32, 64, 128, 256], [32, 64, 128, 256], 0.7, 50, 16, 0.001, (), <attribute '__dict__' of 'BaseConfig' objects>, <attribute '__weakref__' of 'BaseConfig' objects>, None])


In [14]:
%run "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/scripts/data_preprocessing.py"


Found 1000 matched image/mask pairs.
Remapping masks from 59 original classes to 9 superclasses.
Split -> train: 800, val: 100, test: 100
train: saved 800 pairs to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/1_processed/train
val: saved 100 pairs to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/1_processed/val
test: saved 100 pairs to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/1_processed/test
Wrote 9-superclass labels.csv to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/1_processed/labels.csv

Done. Processed data (9-superclass masks) is at: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/1_processed


In [15]:
%run "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/scripts/data_augmentation.py"


train: saved 2400 pairs (800 original + 1600 augmented) to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/2_augmented/train
val: copied 100 pairs unchanged to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/2_augmented/val
test: copied 100 pairs unchanged to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/2_augmented/test

Done. Augmented data is at: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/2_augmented


In [16]:
import tensorflow as tf
import gc

# 1. Clear the Keras global session
tf.keras.backend.clear_session()

# 2. Force garbage collection
gc.collect()


0

In [17]:
%run "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/scripts/train.py" \
    --data-dir 2_augmented \
    --run-name unet_pp_v4_9classes \
    --epochs 60 \
    --batch-size 32 \
    --lr 1e-3 \
    --log-every-n-steps 100 \
    --early-stopping-patience 10 \
    --architecture unet_plus_plus
  #  --resume-from "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_v1/best_model.weights.h5"

Training data: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/2_augmented/train
Validation data: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/2_augmented/val
Train pairs: 2400, Val pairs: 100
Computing class weights from a sample of training masks...
Building architecture: unet_plus_plus
Model param count: 2,210,793
Epoch 1/60
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4210 - loss: 0.7319 - mean_iou: 0.1281
Epoch 1: val_mean_iou improved from None to 0.04140, saving model to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v4_9classes/best_model.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v4_9classes/best_model.weights.h5
75/75 ━━━━━━━━━━━━━━━━━━━━ 226s 1s/step - accuracy: 0.5645 - loss: 0.6567 - mean_iou: 0.1831 - val_accuracy: 0.2710 - val_loss: 0.8393 - val_mean_iou: 0.0414 - learning_rate: 0.0010
Epoch 2/60
75/75 ━━━━━━━━━━━━━━━━━━━━ 

In [18]:
%run "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/scripts/eval.py" \
    --run-name unet_pp_v4_9classes \
    --num-samples 12 \
    --checkpoint "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v4_9classes/best_model.weights.h5"

Loading checkpoint: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v4_9classes/best_model.weights.h5
Evaluating on: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/Data/1_processed/test
Test pairs: 100
Building architecture: unet_plus_plus
Overall mIoU:    0.5533
Foreground mIoU: 0.5017
Mean Dice:       0.6746
Pixel accuracy:  0.9067
Saved metrics to: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v4_9classes/eval/metrics.json
Saved prediction visuals to: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v4_9classes/eval/sample_predictions.png


<Figure size 640x480 with 0 Axes>

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/scripts/inference.py" \
    --image /content/000b3a87508b0fa185fbd53ecbe2e4c6.jpg \
    --run-name unet_pp_v2 \
    --output  /content/000b3a8750_v2_mask.jpg  \
    --checkpoint "/content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v2/best_model.weights.h5"

Loading checkpoint: /content/drive/MyDrive/Colab Notebooks/clothes-segmentation/models/unet_pp_v1/best_model.weights.h5
Reading image: /content/000b3a87508b0fa185fbd53ecbe2e4c6.jpg
Saved predicted mask to: /content/000b3a87508b0fa185fbd53ecbe2e4c6_mask.png
